我要尝试一下 自己使用 FFN 来求解 H2分子的系统基态

In [8]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import optax
from tqdm import tqdm
from functools import partial

# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. 神经网络 Ansatz (Flax Linen 版本)
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 16
    @nn.compact
    def __call__(self, x):
        x = x.astype(jnp.complex128)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(1, param_dtype=jnp.complex128)(x)
        return x.squeeze(-1)

# ==============================================================================
# 3. 辅助函数（使用全局 ha，并保留 JIT）
# ==============================================================================
@partial(jax.jit, static_argnames=('model',))
def compute_local_energies(params, model, sigma):
    eta, H_sigmaeta = ha.get_conn_padded(sigma)
    logpsi_sigma = model.apply(params, sigma)
    logpsi_eta = model.apply(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)

@partial(jax.jit, static_argnames=('model',))
def estimate_energy(params, model, sigma):
    E_loc = compute_local_energies(params, model, sigma)
    return nk.stats.Stats(
        mean=jnp.mean(E_loc),
        error_of_mean=jnp.sqrt(jnp.var(E_loc) / E_loc.size),
        variance=jnp.var(E_loc)
    )

@partial(jax.jit, static_argnames=('model',))
def energy_and_grad(params, model, samples):
    def loss_fn(p):
        E_loc = compute_local_energies(p, model, samples)
        E = jnp.mean(E_loc)
        return E.real, E
    (_, E), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
    return E, grads

# ==============================================================================
# 4. 初始化模型、采样器、优化器
# ==============================================================================
model = SingleStateAnsatz(hidden_dim=12)

dummy_input = hi.random_state(jax.random.key(0), size=1)
params = model.init(jax.random.key(21), dummy_input)

# 采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)
sampler_state = sampler.init_state(model, params, seed=1)

optimizer = optax.adam(learning_rate=1e-2)
opt_state = optimizer.init(params)

# ==============================================================================
# 5. VMC 优化循环
# ==============================================================================
n_iters = 800
for i in tqdm(range(n_iters)):
    sampler_state = sampler.reset(model.apply, params, state=sampler_state)
    samples, sampler_state = sampler.sample(
        model.apply, params, state=sampler_state, chain_length=200
    )
    samples = samples.reshape(-1, hi.size)  # 实际上 reshape 不是必须的

    E, E_grad = energy_and_grad(params, model, samples)   # 注意：不再传入 ha

    updates, opt_state = optimizer.update(E_grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    if i % 50 == 0:
        print(f"Step {i:3d}: Energy = {E.mean():.6f} Ha")

# 最终结果
# 重新采样一组最终样本用于评估
sampler_state = sampler.reset(model.apply, params, state=sampler_state)
samples, _ = sampler.sample(model.apply, params, state=sampler_state, chain_length=200)
samples = samples.reshape(-1, hi.size)
final_energy = estimate_energy(params, model, samples)

print("\n" + "="*60)
print(f"FCI 精确基态能量: {E_fcis[0]:.8f} Ha")
print(f"FFN-VMC 最终能量: {final_energy.mean.real:.6f} ± {final_energy.error_of_mean:.6f} Ha")
print(f"绝对误差: {abs(final_energy.mean.real - E_fcis[0]):.6f} Ha")
print("="*60)

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


  0%|          | 2/800 [00:00<04:45,  2.79it/s]

Step   0: Energy = -0.360868-0.001033j Ha


  6%|▋         | 51/800 [00:07<01:46,  7.02it/s]

Step  50: Energy = -0.310893-0.001745j Ha


...

Step 700: Energy = -0.915001-0.000576j Ha


 94%|█████████▍| 752/800 [01:38<00:06,  7.64it/s]

Step 750: Energy = -0.932165+0.000686j Ha


100%|██████████| 800/800 [01:45<00:00,  7.61it/s]



FCI 精确基态能量: -1.01546825 Ha
FFN-VMC 最终能量: -0.914715 ± 0.004402 Ha
绝对误差: 0.100753 Ha


In [11]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import optax
from tqdm import tqdm
from functools import partial

# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. 神经网络 Ansatz (Flax Linen 版本)
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 16
    @nn.compact
    def __call__(self, x):
        x = x.astype(jnp.complex128)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(1, param_dtype=jnp.complex128)(x)
        return x.squeeze(-1)

# ==============================================================================
# 3. 辅助函数（使用全局 ha，并保留 JIT）
# ==============================================================================
@partial(jax.jit, static_argnames=('model',))
def compute_local_energies(params, model, sigma):
    eta, H_sigmaeta = ha.get_conn_padded(sigma)
    logpsi_sigma = model.apply(params, sigma)
    logpsi_eta = model.apply(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)

@partial(jax.jit, static_argnames=('model',))
def estimate_energy(params, model, sigma):
    E_loc = compute_local_energies(params, model, sigma)
    return nk.stats.Stats(
        mean=jnp.mean(E_loc),
        error_of_mean=jnp.sqrt(jnp.var(E_loc) / E_loc.size),
        variance=jnp.var(E_loc)
    )

# @partial(jax.jit, static_argnames=('model',))
# def energy_and_grad(params, model, samples):
#     def loss_fn(p):
#         E_loc = compute_local_energies(p, model, samples)
#         E = jnp.mean(E_loc)
#         return E.real, E
#     (_, E), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
#     return E, grads


# ==================== 修正后的 energy_and_grad（VJP方式）====================
@partial(jax.jit, static_argnames=('model',))
def energy_and_grad(params, model, samples):
    samples = samples.reshape(-1, samples.shape[-1])
    # 计算 local energies
    eta, H_sigmaeta = ha.get_conn_padded(samples)
    logpsi_s = model.apply(params, samples)
    logpsi_eta = model.apply(params, eta)
    logpsi_s = jnp.expand_dims(logpsi_s, -1)
    E_loc = jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - logpsi_s), axis=-1)  # 复数
    E_avg = jnp.mean(E_loc)
    # 统计量
    E_stats = nk.stats.Stats(
        mean=E_avg,
        error_of_mean=jnp.sqrt(jnp.var(E_loc) / E_loc.size),
        variance=jnp.var(E_loc)
    )
    # 官方 VJP 梯度
    def logpsi_fun(p):
        return model.apply(p, samples)
    _, vjp_fun = jax.vjp(logpsi_fun, params)
    # 注意：这里使用 (E_loc - E_avg) 的共轭？NetKet 内部用的是共轭，但直接这样也可
    grad = vjp_fun((E_loc - E_avg) / E_loc.size)[0]
    return E_stats, grad

# ==================== 超参数调整 ====================
n_chains = 32            # 原来 16
sweep_size = 64          # 原来 32
chain_length = 500       # 原来 200 (每个链的样本数)
n_iters = 800
learning_rate = 0.002    # 原来 0.01

# 初始化参数时缩小标准差
def init_params(key, model, sample_input):
    from flax.core import init
    def init_fn(rng):
        return model.init(rng, sample_input)
    # 自定义初始化，减小 Dense 层的 std
    # 更简单：使用 model.init 后手动缩放
    params = model.init(key, sample_input)
    # 递归缩小参数（仅示例，实际需遍历）
    return params

# 改用上面新的 energy_and_grad

# ==============================================================================
# 4. 初始化模型、采样器、优化器
# ==============================================================================
model = SingleStateAnsatz(hidden_dim=12)

dummy_input = hi.random_state(jax.random.key(0), size=1)
params = model.init(jax.random.key(21), dummy_input)

# 采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)
sampler_state = sampler.init_state(model, params, seed=1)

optimizer = optax.adam(learning_rate=1e-2)
opt_state = optimizer.init(params)

# ==============================================================================
# 5. VMC 优化循环
# ==============================================================================
n_iters = 800
for i in tqdm(range(n_iters)):
    sampler_state = sampler.reset(model.apply, params, state=sampler_state)
    samples, sampler_state = sampler.sample(
        model.apply, params, state=sampler_state, chain_length=200
    )
    samples = samples.reshape(-1, hi.size)  # 实际上 reshape 不是必须的

    E, E_grad = energy_and_grad(params, model, samples)   # 注意：不再传入 ha

    updates, opt_state = optimizer.update(E_grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    if i % 50 == 0:
        print(f"Step {i:3d}: Energy = {E.mean} Ha")

# 最终结果
# 重新采样一组最终样本用于评估
sampler_state = sampler.reset(model.apply, params, state=sampler_state)
samples, _ = sampler.sample(model.apply, params, state=sampler_state, chain_length=200)
samples = samples.reshape(-1, hi.size)
final_energy = estimate_energy(params, model, samples)

print("\n" + "="*60)
print(f"FCI 精确基态能量: {E_fcis[0]:.8f} Ha")
print(f"FFN-VMC 最终能量: {final_energy.mean.real:.6f} ± {final_energy.error_of_mean:.6f} Ha")
print(f"绝对误差: {abs(final_energy.mean.real - E_fcis[0]):.6f} Ha")
print("="*60)

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


  0%|          | 2/800 [00:00<05:07,  2.59it/s]

Step   0: Energy = (-0.36086805684290385-0.001033423146332773j) Ha


  6%|▋         | 52/800 [00:07<01:36,  7.74it/s]

Step  50: Energy = (-0.6534848750930121-0.004960775216755163j) Ha


...

Step 700: Energy = (-0.5991595656377815-0.00020124109706848905j) Ha


 94%|█████████▍| 752/800 [01:41<00:06,  7.22it/s]

Step 750: Energy = (-0.6211522277792206+0.0005079202626682876j) Ha


100%|██████████| 800/800 [01:47<00:00,  7.41it/s]



FCI 精确基态能量: -1.01546825 Ha
FFN-VMC 最终能量: -0.579033 ± 0.004556 Ha
绝对误差: 0.436435 Ha


In [2]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import optax
from tqdm import tqdm
from functools import partial

# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. 神经网络 Ansatz (Flax Linen 版本)
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 8
    @nn.compact
    def __call__(self, x):
        x = x.astype(jnp.complex128)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(1, param_dtype=jnp.complex128)(x)
        return x.squeeze(-1)

# ==============================================================================
# 3. 辅助函数（修正共轭梯度 + 自然梯度支持）
# ==============================================================================
@partial(jax.jit, static_argnames=('model',))
def compute_local_energies(params, model, sigma):
    eta, H_sigmaeta = ha.get_conn_padded(sigma)
    logpsi_sigma = model.apply(params, sigma)
    logpsi_eta = model.apply(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)

@partial(jax.jit, static_argnames=('model',))
def energy_and_grad(params, model, samples):
    samples = samples.reshape(-1, samples.shape[-1])
    # 计算 local energies
    eta, H_sigmaeta = ha.get_conn_padded(samples)
    logpsi_s = model.apply(params, samples)
    logpsi_eta = model.apply(params, eta)
    logpsi_s = jnp.expand_dims(logpsi_s, -1)
    E_loc = jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - logpsi_s), axis=-1)  # 复数
    E_avg = jnp.mean(E_loc)
    # 统计量
    E_stats = nk.stats.Stats(
        mean=E_avg,
        error_of_mean=jnp.sqrt(jnp.var(E_loc) / E_loc.size),
        variance=jnp.var(E_loc)
    )
    # 梯度 VJP，注意共轭
    def logpsi_fun(p):
        return model.apply(p, samples)
    _, vjp_fun = jax.vjp(logpsi_fun, params)
    # 关键修正：对 (E_loc - E_avg) 取共轭
    grad = vjp_fun(jnp.conj(E_loc - E_avg) / E_loc.size)[0]
    return E_stats, grad

# ==============================================================================
# 4. 初始化模型、采样器、优化器（使用自然梯度 SR）
# ==============================================================================
model = SingleStateAnsatz(hidden_dim=12)

dummy_input = hi.random_state(jax.random.key(0), size=1)
params = model.init(jax.random.key(21), dummy_input)

# 采样器（保持你的设置）
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(
    hi, rule=single_rule, n_chains=16, sweep_size=32
)
sampler_state = sampler.init_state(model.apply, params, seed=1)

# 优化器：自然梯度 SR（代替 Adam）
optimizer = nk.optimizer.SR(learning_rate=0.1, diag_shift=0.01)

# ==============================================================================
# 5. VMC 优化循环
# ==============================================================================
n_iters = 500
chain_length = 500   # 每个链的样本数，总样本 = n_chains * chain_length
n_samples = sampler.n_chains * chain_length
print(f"每个迭代的样本数: {n_samples}")
energy_history = []
for i in tqdm(range(n_iters)):
    # 重置采样器（参数变化后必须重置）
    sampler_state = sampler.reset(model.apply, params, state=sampler_state)
    samples, sampler_state = sampler.sample(
        model.apply, params, state=sampler_state, chain_length=chain_length
    )
    samples = samples.reshape(-1, hi.size)

    # 计算能量和自然梯度
    E_stats, grad = energy_and_grad(params, model, samples)
    # update parameters
    params = jax.tree.map(lambda x,y: x-0.01*y, params, grad)
    energy_history.append(E_stats.mean.real)

    if i % 10 == 0:
        print(f"Step {i:3d}: Energy = {E_stats.mean.real:.6f} ± {E_stats.error_of_mean:.6f} Ha")

# 最终评估：重新采样一批独立样本
sampler_state = sampler.reset(model.apply, params, state=sampler_state)
samples, _ = sampler.sample(
    model.apply, params, state=sampler_state, chain_length=2000
)
samples = samples.reshape(-1, hi.size)
final_stats = energy_and_grad(params, model, samples)[0]

print("\n" + "="*60)
print(f"FCI 精确基态能量: {E_fcis[0]:.8f} Ha")
print(f"FFN-VMC 最终能量: {final_stats.mean.real:.6f} ± {final_stats.error_of_mean:.6f} Ha")
print(f"绝对误差: {abs(final_stats.mean.real - E_fcis[0]):.6f} Ha")
print("="*60)

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV
每个迭代的样本数: 8000


  0%|          | 1/500 [00:00<05:46,  1.44it/s]

Step   0: Energy = -0.361513 ± 0.002285 Ha


  2%|▏         | 11/500 [00:04<03:36,  2.26it/s]

Step  10: Energy = -0.423729 ± 0.002962 Ha


...

Step 480: Energy = -0.646696 ± 0.003189 Ha


 98%|█████████▊| 491/500 [02:38<00:03,  2.51it/s]

Step 490: Energy = -0.649453 ± 0.003248 Ha


100%|██████████| 500/500 [02:40<00:00,  3.11it/s]



FCI 精确基态能量: -1.01546825 Ha
FFN-VMC 最终能量: -0.658179 ± 0.001585 Ha
绝对误差: 0.357289 Ha


In [23]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import optax
from tqdm import tqdm

# ==============================================================================
# 1. 全局参数 & H₂ 分子定义（保持不变）
# ==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. 神经网络 Ansatz（保持不变）
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 8
    @nn.compact
    def __call__(self, x):
        x = x.astype(jnp.complex128)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(1, param_dtype=jnp.complex128)(x)
        return x.squeeze(-1)

# ==============================================================================
# 3. 辅助函数（简化，保留核心逻辑）
# ==============================================================================
@jax.jit
def compute_energy_and_grad(params, model, samples):
    samples = samples.reshape(-1, samples.shape[-1])
    
    # 1. 计算 Local Energy
    eta, H_sigmaeta = ha.get_conn_padded(samples)
    logpsi_s = model.apply(params, samples)
    logpsi_eta = model.apply(params, eta)
    logpsi_s = jnp.expand_dims(logpsi_s, -1)
    E_loc = jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - logpsi_s), axis=-1)
    
    # 2. 统计量
    E_mean = jnp.mean(E_loc).real
    E_var = jnp.var(E_loc).real
    E_error = jnp.sqrt(E_var / E_loc.size)
    
    # 3. 普通梯度 (VJP)
    def logpsi_fun(p):
        return model.apply(p, samples)
    _, vjp_fun = jax.vjp(logpsi_fun, params)
    grad = vjp_fun(jnp.conj(E_loc - E_mean) / E_loc.size)[0]
    
    return E_mean, E_error, grad

# ==============================================================================
# 4. 初始化：SR预条件器 + SGD优化器（核心修改部分）
# ==============================================================================
model = SingleStateAnsatz(hidden_dim=12)
dummy_input = hi.random_state(jax.random.key(0), size=1)
params = model.init(jax.random.key(21), dummy_input)

# 采样器（保持不变）
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(
    hi, rule=single_rule, n_chains=16, sweep_size=32
)
sampler_state = sampler.init_state(model.apply, params, seed=1)

# -------------------------- 核心修改开始 --------------------------
# 1. 基础优化器：SGD
lr = 0.05
optimizer = optax.sgd(learning_rate=lr)
opt_state = optimizer.init(params)

# 2. SR预条件器（注意：这不是optimizer，是preconditioner）
sr = nk.optimizer.SR(
    diag_shift=0.01,                        # 正则化，防止奇异
    holomorphic=False                       # 波函数是复数但非全纯（实参数）
)
#sr_state = sr.init(params)  # 初始化SR状态
# -------------------------- 核心修改结束 --------------------------

# ==============================================================================
# 5. VMC 优化循环（核心修改部分）
# ==============================================================================
n_iters = 300
chain_length = 500
n_samples = sampler.n_chains * chain_length
print(f"\n每个迭代的样本数: {n_samples}")
print("="*60)

energy_history = []

for i in tqdm(range(n_iters)):
    # 1. 采样
    sampler_state = sampler.reset(model.apply, params, state=sampler_state)
    samples, sampler_state = sampler.sample(
        model.apply, params, state=sampler_state, chain_length=chain_length
    )
    
    # 2. 计算能量和普通梯度
    E_mean, E_error, grad = compute_energy_and_grad(params, model, samples)
    energy_history.append(E_mean)
    
    # -------------------------- 核心修改开始 --------------------------
    # 3. 应用SR预条件：将普通梯度 -> 自然梯度方向
    updates, sr_state = sr.apply(
        sr_state, 
        params, 
        grad, 
        model_apply=model.apply, 
        samples=samples
    )
    
    # 4. 用SGD优化器应用更新
    updates, opt_state = optimizer.update(updates, opt_state, params)
    params = optax.apply_updates(params, updates)
    # -------------------------- 核心修改结束 --------------------------
    
    # 打印
    if i % 10 == 0:
        print(f"Step {i:3d} | Energy = {E_mean:.6f} ± {E_error:.6f} Ha")

# ==============================================================================
# 6. 最终评估
# ==============================================================================
sampler_state = sampler.reset(model.apply, params, state=sampler_state)
samples, _ = sampler.sample(
    model.apply, params, state=sampler_state, chain_length=2000
)
final_E, final_err, _ = compute_energy_and_grad(params, model, samples)

print("\n" + "="*60)
print(f"FCI 精确基态能量: {E_fcis[0]:.8f} Ha")
print(f"SR+SGD-VMC 最终能量: {final_E:.6f} ± {final_err:.6f} Ha")
print(f"绝对误差: {abs(final_E - E_fcis[0]):.6f} Ha")
print("="*60)

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV

每个迭代的样本数: 8000


  0%|          | 0/300 [00:00<?, ?it/s]


TypeError: Error interpreting argument to <function compute_energy_and_grad at 0x342357b00> as an abstract array. The problematic value is of type <class '__main__.SingleStateAnsatz'> and was passed to the function at path model.
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.